<a href="https://colab.research.google.com/github/amoghkamath06/Bio_Inspired_system/blob/main/cuckoo_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import random

# Processing times of 3 jobs on 3 machines
jobs = [
    [3, 2, 2],  # Job 0 times on Machine 0, 1, 2
    [2, 1, 4],  # Job 1 times on Machine 0, 1, 2
    [4, 3, 1]   # Job 2 times on Machine 0, 1, 2
]

# Calculate completion time (makespan)
def fitness(schedule):
    machine_time = [0, 0, 0]
    job_time = [0, 0, 0]

    for job in schedule:
        for m in range(3):
            start = max(machine_time[m], job_time[job])
            finish = start + jobs[job][m]

            machine_time[m] = finish
            job_time[job] = finish

    return max(machine_time)

# Generate random schedule
def new_schedule():
    schedule = [0, 1, 2] * 3
    random.shuffle(schedule)
    return schedule

# Modify schedule using DCS operators
def modify(schedule):
    s = schedule.copy()
    a, b = random.sample(range(len(s)), 2)

    operation = random.choice([1, 2, 3])

    if operation == 1:
        value = s.pop(a)
        s.insert(b, value)

    elif operation == 2:
        s[a], s[b] = s[b], s[a]

    else:
        if a > b:
            a, b = b, a
        s[a:b+1] = s[a:b+1][::-1]

    return s

# Helper function to print a detailed, readable timeline of the schedule
def print_detailed_schedule(schedule):
    machine_time = [0, 0, 0]
    job_time = [0, 0, 0]
    job_task_index = [0, 0, 0]

    print("\n--- Detailed Schedule Timeline ---")
    for i, job in enumerate(schedule):
        task = job_task_index[job]
        print(f"\nStep {i+1}: Schedule Job {job} (Task {task} on Machine {task})")
        for m in range(3):
            start = max(machine_time[m], job_time[job])
            duration = jobs[job][m]
            finish = start + duration

            print(f"  - Machine {m}: start = max(M{m}_free={machine_time[m]}, Job_{job}_free={job_time[job]}) = {start} -> ends at {finish} (Duration: {duration})")

            machine_time[m] = finish
            job_time[job] = finish
        job_task_index[job] += 1

    print("\nFinal Machine End Times:")
    for m in range(3):
        print(f"  Machine {m} finished at: {machine_time[m]}")

# Discrete Cuckoo Search
nests = [new_schedule() for _ in range(10)]

best_values = []

for iteration in range(50):

    for i in range(len(nests)):
        new = modify(nests[i])

        if fitness(new) < fitness(nests[i]):
            nests[i] = new

    # Abandon some poor solutions
    nests.sort(key=fitness)

    for i in range(7, 10):
        nests[i] = new_schedule()

    best = min(nests, key=fitness)
    best_values.append(fitness(best))

print("Best Schedule found by Cuckoo Search:", best)
print("Minimum Completion Time (Makespan):", fitness(best))

# Print the step-by-step breakdown of the best schedule
print_detailed_schedule(best)

Best Schedule found by Cuckoo Search: [0, 1, 2, 0, 1, 2, 0, 1, 2]
Minimum Completion Time (Makespan): 31

--- Detailed Schedule Timeline ---

Step 1: Schedule Job 0 (Task 0 on Machine 0)
  - Machine 0: start = max(M0_free=0, Job_0_free=0) = 0 -> ends at 3 (Duration: 3)
  - Machine 1: start = max(M1_free=0, Job_0_free=3) = 3 -> ends at 5 (Duration: 2)
  - Machine 2: start = max(M2_free=0, Job_0_free=5) = 5 -> ends at 7 (Duration: 2)

Step 2: Schedule Job 1 (Task 0 on Machine 0)
  - Machine 0: start = max(M0_free=3, Job_1_free=0) = 3 -> ends at 5 (Duration: 2)
  - Machine 1: start = max(M1_free=5, Job_1_free=5) = 5 -> ends at 6 (Duration: 1)
  - Machine 2: start = max(M2_free=7, Job_1_free=6) = 7 -> ends at 11 (Duration: 4)

Step 3: Schedule Job 2 (Task 0 on Machine 0)
  - Machine 0: start = max(M0_free=5, Job_2_free=0) = 5 -> ends at 9 (Duration: 4)
  - Machine 1: start = max(M1_free=6, Job_2_free=9) = 9 -> ends at 12 (Duration: 3)
  - Machine 2: start = max(M2_free=11, Job_2_free=12) =